# Welcome to NER Finetuning with HF Transformers 

## Contents 
1. Configurations (Resources, Training, Optimization, Evaluation, Saving results)
2. Prepare training & dataset
3. Train (Finetune) a NER Model
4. Optimization
5. Evaluation

In [1]:
# import modules
import config
import train
import eval_opt
import merge_datasets

/data-ssd/sophie.schneider/pyenv/versions/hf-transformers-env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Configurations

In this section, we set all of the configurations for the following steps (training, optimization, evaluation). Depending on which parts you want to run and which to leave out, corresponding config parts can be skipped if not needed.

### Resources

We need to state which resources we want to use for NER finetuning, namely the `dataset` and `model`. Therefore, we use the `Resource` class and define the `path` (either to a HF or a local directory) and `source` (`hf` or `local`) of our resources. By applying `set_name()`, the model or dataset name gets set automatically, otherwise the `model.name` can also be updated manually.

### Load Model

You can choose e.g. one of the following models and add them as `path` in the model config:
* FacebookAI/roberta-base
* dbmdz/bert-base-german-cased
* dbmdz/bert-base-historic-multilingual-cased
* flair/ner-german (not implemented so far)
* distilbert/distilbert-base-uncased

In [2]:
model = config.Resource(path="dbmdz/bert-base-german-europeana-uncased", source="hf")
model.set_name()
print(model.info())

bert-base-german-europeana-uncased will be loaded from dbmdz/bert-base-german-europeana-uncased (via hf).


### Load Dataset

You can choose e.g. one of the following dataset configurations:
* `path="data/zefys2025.hf", source="local"`
* `path="eriktks/conll2003", source="hf"`
* `path="GermanEval/germeval_14", source="hf"`
* `path="data/hisgermaner.hf", source="local"`
* `path="data/newseye.hf", source="local"`
* `path="data/hipe2020.hf", source="local"`

In [3]:
dataset = config.Resource(path="data/hipe2020.hf", source="local")
dataset.set_name()
print(dataset.info())

hipe2020 will be loaded from data/hipe2020.hf (via local).


In [4]:
train.set_torch_device()

device(type='cuda')

## 2. Prepare dataset

In [5]:
ner_dataset = train.load_ner_dataset(dataset.path, dataset.source)
ner_dataset["train"][4]

{'id': 4,
 'tokens': ['Auf',
  'Talot',
  "'",
  's',
  'Vorschlag',
  'beschloß',
  'man',
  'über',
  'den',
  'constitutionellen',
  'Umkreis',
  ',',
  'welchen',
  'das',
  'gesetzgebende',
  'Corps',
  'in',
  'Zukunft',
  'inne',
  'haben',
  'soll',
  ',',
  'fol',
  '¬',
  'gendes',
  ':',
  'Vom',
  'Tage',
  'an',
  ',',
  'da',
  'der',
  'Rath',
  'der',
  '500',
  'in',
  'sei',
  '¬',
  'nen',
  'neuen',
  'Pallast',
  'installirt',
  'seyn',
  'wird',
  ',',
  'sind',
  'die',
  'äußerlichen',
  'Bezirke',
  'für',
  'beyde',
  'Räthe',
  'folgendermas',
  '¬',
  'sen',
  'firir',
  ':',
  'Rarh',
  'der',
  'Alten',
  ':',
  'Der',
  'Umfang',
  'des',
  'Na',
  '¬',
  'tionalpallastes',
  'der',
  'Alten',
  ',',
  'in',
  'den',
  'Tuillerien',
  'situirt',
  ',',
  'enhält',
  'gegen',
  'Westen',
  'die',
  'Straße',
  'und',
  'den',
  'Platz',
  'des',
  'Carrousel',
  'bis',
  'zum',
  'Eintritt',
  'in',
  'die',
  'Straße',
  'Nicaise',
  ',',
  'am',
  'Hause

In [ ]:
"""
# for hipe2020
from datasets import load_dataset
ner_dataset = load_dataset('bigscience-historical-texts/HIPE2020_sent-split', "de")
ner_dataset["train"][1]
"""

In [37]:
"""
# for roberta
import torch
from datetime import datetime
from datasets import load_dataset, load_from_disk
from transformers import AutoTokenizer, AutoModelForTokenClassification, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer
from transformers import DataCollatorForTokenClassification
import eval_opt
import evaluate
import numpy as np

def get_tokenizer(model_path):
    tokenizer = AutoTokenizer.from_pretrained(model_path, add_prefix_space=True)
    return tokenizer

tokenizer = get_tokenizer(model.path)
"""

In [6]:
tokenizer = train.get_tokenizer(model.path)

In [7]:
tokenized_dataset = train.prepare_dataset(ner_dataset, tokenizer)
label_list = train.get_label_list(ner_dataset)

Map: 100%|█████████████████████████████████████████████████████████████████| 1202/1202 [00:00<00:00, 9966.56 examples/s]


In [8]:
label_list

['I-PROD',
 'B-LOC',
 'B-TIME',
 'B-PROD',
 'O',
 'B-ORG',
 'I-ORG',
 'B-PER',
 'I-TIME',
 'I-PER',
 'I-LOC']

## [optional] 2a. Remove specific labels from Dataset
below: how to remove labels that do not occur in our zefys2025 dataset?

In [5]:
ner_dataset = train.load_ner_dataset(dataset.path, dataset.source)
label_list = train.get_label_list(ner_dataset)
ner_dataset["train"].features["ner_tags"]

Sequence(feature=ClassLabel(names=['I-PROD', 'B-LOC', 'B-TIME', 'B-PROD', 'O', 'B-ORG', 'I-ORG', 'B-PER', 'I-TIME', 'I-PER', 'I-LOC'], id=None), length=-1, id=None)

In [6]:
dataset_updated = merge_datasets.drop_ner_labels(label_list, ner_dataset)

Casting the dataset: 100%|████████████████████████████████████████████████| 1202/1202 [00:00<00:00, 17128.97 examples/s]


In [7]:
#as we can see now, the labels from tokenized dataset that are not in our zefys_label_list are removed from the ClassLabels (replaced with "O")
print(dataset_updated["train"].features["ner_tags"])

Sequence(feature=ClassLabel(names=['B-LOC', 'O', 'B-ORG', 'I-ORG', 'B-PER', 'I-PER', 'I-LOC'], id=None), length=-1, id=None)


In [9]:
tokenizer = train.get_tokenizer(model.path)
tokenized_dataset = train.prepare_dataset(dataset_updated, tokenizer)
label_list = train.get_label_list(dataset_updated)

Map: 100%|████████████████████████████████████████████████████████████████| 1215/1215 [00:00<00:00, 10446.94 examples/s]


## [optional] 2b. Merge multiple datasets
https://huggingface.co/docs/datasets/v1.1.1/processing.html#concatenate-several-datasets

In [ ]:
#should be similar to the following (from HF docs - link above, not tested yet):
merged_dataset = concatenate_datasets([dataset1, dataset2])
#DatasetDict({"train": datasets.concatenate_datasets([dd1["train"], dd2["train"]])})

## 3. Train (Finetune) a NER Model

To finetune our model for NER, we first need to define our `ner` task (see train.py). 

### Training Parameters

You can define a variable for the training parameters based on our default `TrainingParams` config and try to improve results using evaluation/optimization techniques, or start by overwriting the parameters for training manually. Training will only be performed once on the training dataset and evaluated once on the validation dataset. For more sophisticated methods, you can additionally run the Optimization and Evaluation parts to find ideal hyperparameter combinations and perform a more reliable evaluation.

In [ ]:
training_params = config.TrainingParams()
training_params.__dict__

In [ ]:
training_params.batch_size = 32
training_params.num_train_epochs = 1
training_params.__dict__

In [ ]:
trained_ner_model, model_out_path = train.train_model(model, dataset, label_list, training_params, tokenized_dataset, tokenizer)

## 4. Optimization

### Evaluation and optimization

* Evaluation: Stratified k-fold Cross-Validation - one of ["None", k] (https://huggingface.co/docs/datasets/v1.11.0/splits.html#examples; https://discuss.huggingface.co/t/k-fold-cross-validation/5765/5, https://github.com/huggingface/datasets/discussions/5940)
* Optimization:
  * https://huggingface.co/docs/transformers/hpo_train
  * https://medium.com/carbon-consulting/transformer-models-hyperparameter-optimization-with-the-optuna-299e185044a8
  * Grid Search (GridSearchCV?), Line Search, hyperopt
* Analyze grid search results using Heatmap or similar

### Optimization

For optimization (hyperparameter search) we use the HF Trainer API (https://huggingface.co/docs/transformers/hpo_train#hyperparameter-search-using-trainer-api) and optuna (https://optuna.readthedocs.io/en/stable/index.html, `pip install optuna`). We can access all of the search parameters, as they are stored in a dict structure, and also adapt the number of trials `n_trials` to run. 

In [ ]:
optimize_params = config.OptimizeParams()

optimize_params.hp_space["learning_rate"]["param_type"]
optimize_params.n_trials = 10
optimize_params.__dict__

In [ ]:
#to remove an hyperparameter from the hp_space dict, use pop() or similar
optimize_params.hp_space.pop("warmup_steps")
optimize_params.__dict__

In [ ]:
model_out_path = "roberta_germeval14_opt/"
eval_opt.optimize(optimize_params, training_params, model.path, model_out_path, label_list, tokenizer, tokenized_dataset)

## 5. Evaluation

In [ ]:
class_report, errors = eval_opt.compute_metrics_per_tag(trained_ner_model, tokenized_dataset, label_list) 

In [ ]:
eval_opt.save_class_report(class_report, "md", model_out_path)

In [ ]:
config.save_train_config(model_out_path, training_params)

In [ ]:
#inference example
from transformers import pipeline, AutoModelForTokenClassification

text = "Novelle	von Joh. L. Buchta."

id2label = {0:label_list[0], 1:label_list[1], 2:label_list[2], 3:label_list[3], 4:label_list[4], 5:label_list[5], 6:label_list[6]}
finetuned_model = AutoModelForTokenClassification.from_pretrained("xlm-roberta-base_hisgermaner_2025-04-02_14-45/checkpoint-276", num_labels=7, id2label=id2label)
classifier = pipeline("ner", model=finetuned_model, tokenizer=tokenizer)
classifier(text)